In [1]:
!pip install pennylane scipy matplotlib pyscf

In [1]:
#cond  4.507766x10^12
import os
import pennylane as qml
from pennylane import numpy as np
import matplotlib.pyplot as plt
import scipy


def inite(elec,orb):
    config=[]
    list1=[]
    #singles
    for x in range(elec):
        count=orb-elec
        while (count<orb):
            for e in range(elec):
                if x==e:
                    if x%2==0:
                        config.append(count)
                        count=count+2
                    else:
                        config.append(count+1)
                        count=count+2
                else:
                    config.append(e)
                
            list1.append(config)
            config=[]
    #doubles
    for x in range(elec):
        for y in range(x+1,elec):
            count1=orb-elec
            count2=orb-elec
            for count1 in range(elec, orb, 2):
                for count2 in range(elec, orb, 2):
                    cont=0
                    if count1==count2:
                        if (x%2)!=(y%2):
                            cont=1
                    else:
                        cont=1
                    if (x%2)==(y%2) and count2<count1:
                        cont=0
                    if cont==1:    
                        for e in range(elec):
                            if x==e:
                                if x%2==0:
                                    config.append(count1)
                                else:
                                    config.append(count1+1)
                            elif y==e:
                                if y%2==0:
                                    config.append(count2)
                                else:
                                    config.append(count2+1)
                            else:
                                config.append(e)

                        list1.append(config)
                        config=[]
    return list1

def gs_exact(symbols, geometry, active_electrons, active_orbitals, charge, shots=None, max_iter=100, out_file="gs_uccsd_params.txt"):
    # Build the electronic Hamiltonian
    H, qubits = qml.qchem.molecular_hamiltonian(symbols, geometry, basis="sto-6g", method="pyscf", active_electrons=active_electrons, active_orbitals=active_orbitals, charge=charge)
    hf_state = qml.qchem.hf_state(active_electrons, qubits)
    singles, doubles = qml.qchem.excitations(active_electrons, qubits)
    # Map excitations to the wires the UCCSD circuit will act on
    s_wires, d_wires = qml.qchem.excitations_to_wires(singles, doubles)

    wires=range(qubits)
    params = np.zeros(len(singles) + len(doubles))
    # Define the device
    dev = qml.device("default.qubit", wires=qubits, shots=shots)
    
    # Define the qnode
    if shots==None:
        @qml.qnode(dev, interface="autograd", diff_method="adjoint")
        def circuit(params, wires, s_wires, d_wires, hf_state):
            qml.UCCSD(params, wires, s_wires, d_wires, hf_state)
            return qml.expval(H)
    else:
        @qml.qnode(dev, interface="autograd")
        def circuit(params, wires, s_wires, d_wires, hf_state):
            qml.UCCSD(params, wires, s_wires, d_wires, hf_state)
            return qml.expval(H)

    optimizer = qml.GradientDescentOptimizer(stepsize=2.0)
    for n in range(max_iter):
        params, energy = optimizer.step_and_cost(circuit, params, wires=range(qubits),
                                                 s_wires=s_wires, d_wires=d_wires,
                                                 hf_state=hf_state)

    # Save optimized UCCSD parameters with operator labels
    # params are ordered as: [singles..., doubles...] aligned with `singles`/`doubles` from qml.qchem.excitations
    pr = []
    pr.append("UCCSD parameters (ground state)")
    pr.append("Operator | Parameter")

    # Singles: a^ i (create in virtual a, annihilate in occupied i)
    for k, (i, a) in enumerate(singles):
        pr.append(f"{a}^, {i}\t| {params[k].item()}")

    # Doubles: a^ b^ j i (create in virtual a,b; annihilate in occupied i,j)
    offset = len(singles)
    for k, (i, j, a, b) in enumerate(doubles):
        pr.append(f"{a}^, {i}; {b}^, {j}\t| {params[offset + k].item()}")

    if out_file is not None:
        with open(out_file, "w") as f:
            for line in pr:
                f.write(line + "\n")

    #print(circuit(params, wires, s_wires, d_wires, hf_state))
    return params
def ee_exact(symbols, geometry, active_electrons,  active_orbitals, charge,params,shots=0):
    # Build the electronic Hamiltonian
    H, qubits = qml.qchem.molecular_hamiltonian(symbols, geometry, basis="sto-6g", method="pyscf", active_electrons=active_electrons, active_orbitals=active_orbitals, charge=charge)
    singles, doubles = qml.qchem.excitations(active_electrons, qubits)
    # Map excitations to the wires the UCCSD circuit will act on
    s_wires, d_wires = qml.qchem.excitations_to_wires(singles, doubles)
    wires=range(qubits)

    null_state = np.zeros(qubits,int)
    list1 = inite(active_electrons,qubits)
    values =[]
    for t in range(1):
        if shots==0:
            #dev = qml.device("default.qubit", wires=qubits)
            dev = qml.device("default.qubit", wires=qubits)
        else:
            #dev = qml.device("default.qubit", wires=qubits,shots=shots)
            dev = qml.device("default.qubit", wires=qubits,shots=shots)
        #circuit for diagonal part
        @qml.qnode(dev)
        def circuit_d(params, occ,wires, s_wires, d_wires, hf_state):
            for w in occ:
                qml.X(wires=w)
            qml.UCCSD(params, wires, s_wires, d_wires, hf_state)
            return qml.expval(H)
        #circuit for off-diagonal part
        @qml.qnode(dev)
        def circuit_od(params, occ1, occ2,wires, s_wires, d_wires, hf_state):
            for w in occ1:
                qml.X(wires=w)
            first=-1
            for v in occ2:
                if v not in occ1:
                    if first==-1:
                        first=v
                        qml.Hadamard(wires=v)
                    else:
                        qml.CNOT(wires=[first,v])
            for v in occ1:
                if v not in occ2:
                    if first==-1:
                        first=v
                        qml.Hadamard(wires=v)
                    else:
                        qml.CNOT(wires=[first,v])
            qml.UCCSD(params, wires, s_wires, d_wires, hf_state)
            return qml.expval(H)
        #final M matrix
        M = np.zeros((len(list1),len(list1)))
        for i in range(len(list1)):
            for j in range(len(list1)):
                if i == j:
                    M[i,i] = circuit_d(params, list1[i], wires, s_wires, d_wires, null_state)
        #print("diagonal parts done")
        for i in range(len(list1)):
            for j in range(len(list1)):
                if i!=j:
                    Mtmp = circuit_od(params, list1[i],list1[j],wires, s_wires, d_wires, null_state)
                    M[i,j]=Mtmp-M[i,i]/2.0-M[j,j]/2.0
        #print("off diagonal terms done")
        #ERROR:not subtracting the gs energy
        eig,evec=np.linalg.eig(M)
        values.append(np.sort(eig))
        hf_state = np.array(range(active_electrons))  # Hartree-Fock occupied orbitals

    # Sort eigenvalues and get corresponding eigenvectors
    idx = np.argsort(eig)
    eig = eig[idx]
    evec = evec[:, idx]

    # Write ALL excited states (state 1,2,...) to an output file
    pr = []
    pr.append("R1/R2")
    pr.append("Excited states (eigenvectors of M)")

    for state_idx in range(1, len(eig)):
        vector = evec[:, state_idx]
        pr.append("")
        pr.append("========================================")
        pr.append(f"STATE {state_idx}  E = {eig[state_idx]:.12f}")
        pr.append("========================================")
        pr.append("Excitations | Coefficients")

        for det_idx, coeff in enumerate(vector):
            det = list1[det_idx]
            # Find holes and particles
            holes = [hf for hf in hf_state if hf not in det]
            particles = [virt for virt in det if virt not in hf_state]
            # Print in the style "a^, i" pairs
            s = ""
            for p, h in zip(particles, holes):
                s += f"{p}^, {h}"
            pr.append(f"{s}\t| {coeff.item()}")

    # Save to text file
    with open("eigvecs.txt", "w") as f:
        for line in pr:
            f.write(line + "\n")
    return values

# Define the molecule
r = 1.88973

active_electrons = 4
active_orbitals = 4

charge = 0

# --- Print SCF energy explicitly (PySCF RHF) ---
from pyscf import gto, scf

r_scf = 1.5  # nearest-neighbor distance in Angstrom
mol = gto.M(
    atom=[
        ("H", (0.0 * r_scf*r, 0.0, 0.0)),
        ("H", (1.0 * r_scf*r, 0.0, 0.0)),
        ("H", (2.0 * r_scf*r, 0.0, 0.0)),
        ("H", (3.0 * r_scf*r, 0.0, 0.0)),
    ],
    basis="sto-6g",
    unit="Angstrom",
    charge=charge,
    spin=0,
    verbose=0,
)


active_electrons = 4
active_orbitals = 4

mf = scf.RHF(mol)
e_scf = mf.kernel()
print(f"SCF (RHF) total energy [sto-6g, H4 linear, r={r_scf} Å] = {e_scf:.12f} Ha")

params = gs_exact(symbols,geometry, active_electrons, active_orbitals, charge,max_iter=50)
#params_exact=params

total_energy = ee_exact(symbols, geometry, active_electrons, active_orbitals, charge, params)
#print('exact eigenvalues:\n', total_energy)

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


KeyboardInterrupt: 

### Two-elec integral

In [3]:
from pennylane.qchem import Molecule, electron_integrals
import numpy as np

mult = 1

# --- Compute molecular integrals ---
mol = Molecule(symbols, geometry, charge=charge, mult=mult)
core_constant, one_elec, two_elec = electron_integrals(mol)()

norb = two_elec.shape[0]
nspin = 2 * norb

# Build antisymmetrized spatial tensor <pq||rs>
antisym = np.zeros_like(two_elec)
for p in range(norb):
    for q in range(norb):
        for r in range(norb):
            for s in range(norb):
                antisym[p, q, r, s] = (
                    two_elec[p, q, r, s] - two_elec[p, q, s, r]
                )

lines = []
lines.append("Spin-orbital integrals <pq||rs>:")

for i in range(nspin):
    for j in range(nspin):
        for k in range(nspin):
            for l in range(nspin):

                # Map spin orbitals -> spatial orbitals
                p, q = i // 2, j // 2
                r, s = k // 2, l // 2

                # Spin projection (0 = alpha, 1 = beta)
                si, sj = i % 2, j % 2
                sk, sl = k % 2, l % 2

                # Spin selection rule
                if si == sk and sj == sl:
                    val = antisym[p, q, r, s]
                    if abs(val) > 1e-10:
                        lines.append(f"{i:3d}^ {j:3d}^ {k:3d} {l:3d}  {val:.10f}")

text = "\n".join(lines)

with open("two_elec.txt", "w") as f:
    f.write(text)
